<a href="https://colab.research.google.com/github/Divyaanshvats/Zenteiq_Assignment/blob/main/Zenteiq_qwen3_0_71B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 1 — Install
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd /content/maxtext
!git checkout maxtext-v0.2.2
!pip install -e ".[tpu]" -q

Cloning into 'maxtext'...
remote: Enumerating objects: 97452, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 97452 (delta 97), reused 98 (delta 56), pack-reused 97275 (from 2)
Receiving objects: 100% (97452/97452), 410.64 MiB | 18.92 MiB/s, done.
Resolving deltas: 100% (71518/71518), done.
/content/maxtext
Note: switching to 'maxtext-v0.2.2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 348355c51 Update MaxText version to v0.2.2 and add 

In [ ]:
# CELL 2 — Verify
%cd /content/maxtext/src
!python -c "import jax; print('Devices:', jax.devices())"
# Should show CpuDevice

# CELL 3 — Dirs
!mkdir -p /content/maxtext/runs
!mkdir -p /content/maxtext/logs

# CELL 5 — Train 1.7B (scaled up, max memory savings)
%cd /content/maxtext/src

!python -u maxtext/trainers/pre_train/train.py \
  maxtext/configs/models/qwen3-0.6b.yml \
  steps=50 \
  dataset_type=synthetic \
  hardware=cpu \
  attention=dot_product \
  skip_jax_distributed_system=true \
  base_emb_dim=1152 \
  base_mlp_dim=3456 \
  base_output_directory=/content/maxtext/runs \
  run_name=qwen3_0p71b_cpu_50steps \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  micro_batch_size_to_train_on=1 \
  max_target_length=32 \
  max_prefill_predict_length=32 \
  enable_checkpointing=false \
  scan_layers=false \
  log_period=1 \
  weight_dtype=bfloat16 \
  2>&1 | tee /content/maxtext/logs/qwen3_0p71b_cpu.log

/content/maxtext/src
Failed to find host bounds for accelerator type: WARNING: could not determine TPU accelerator type, please set env var `TPU_ACCELERATOR_TYPE` manually, otherwise libtpu.so may not properly initialize.
E0000 00:00:1781937889.769187    9189 common_lib.cc:554] INVALID_ARGUMENT: Error: unexpected worker hostname 'WARNING: could not determine TPU worker hostnames or IP addresses' from env var TPU_WORKER_HOSTNAMES. Expecting a valid hostname or IP address without port number, or hostname:port:address triple. (Full TPU workers' addr string: WARNING: could not determine TPU worker hostnames or IP addresses, please set env var `TPU_WORKER_HOSTNAMES` manually, otherwise libtpu.so may not properly initialize.)
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/libtpu_init_utils.cc:327
Devices: [CpuDevice(id=0)]
/content/maxtext/src
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-

In [ ]:
# CELL 1 — Install
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd /content/maxtext
!git checkout maxtext-v0.2.2
!pip install -e ".[cuda12]" -q

Cloning into 'maxtext'...
remote: Enumerating objects: 97452, done.
remote: Counting objects: 100% (177/177), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 97452 (delta 97), reused 98 (delta 56), pack-reused 97275 (from 2)
Receiving objects: 100% (97452/97452), 410.64 MiB | 19.20 MiB/s, done.
Resolving deltas: 100% (71518/71518), done.
/content/maxtext
Note: switching to 'maxtext-v0.2.2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 348355c51 Update MaxText version to v0.2.2 and add 

In [ ]:
# CELL 2 — Remove broken transformer_engine + verify
!pip uninstall transformer-engine -y

%cd /content/maxtext/src
!python -c "import jax; print('Devices:', jax.devices())"
# Should show CudaDevice / GpuDevice

# CELL 3 — Dirs
!mkdir -p /content/maxtext/runs
!mkdir -p /content/maxtext/logs

# CELL 4 — Train 0.71B custom scaled model
!python -u maxtext/trainers/pre_train/train.py \
  maxtext/configs/models/qwen3-0.6b.yml \
  steps=50 \
  dataset_type=synthetic \
  hardware=gpu \
  attention=dot_product \
  skip_jax_distributed_system=true \
  base_emb_dim=1152 \
  base_mlp_dim=3456 \
  base_output_directory=/content/maxtext/runs \
  run_name=qwen3_0p71b_gpu_50steps \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  micro_batch_size_to_train_on=1 \
  max_target_length=32 \
  max_prefill_predict_length=32 \
  enable_checkpointing=false \
  scan_layers=false \
  log_period=1 \
  weight_dtype=bfloat16 \
  2>&1 | tee /content/maxtext/logs/qwen3_0p71b_gpu.log

Found existing installation: transformer_engine 2.16.0
Uninstalling transformer_engine-2.16.0:
  Successfully uninstalled transformer_engine-2.16.0
/content/maxtext/src
Devices: [CudaDevice(id=0)]
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
[transformers] DeepseekV32Config got `key=rope_scaling` in kwargs but hasn't set it as attribute. For RoPE standardization you need to set `self.rope_parameters` in model's config. 
I0620 07:23:37.193766 136060562649728 max_utils.py:243] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
I0620 07:23:37.283311 136060562649728 xla_bridge.py:849] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file

In [ ]:
# CELL 1 — Install
!git clone https://github.com/AI-Hypercomputer/maxtext.git
%cd /content/maxtext
!git checkout maxtext-v0.2.2
!pip install -e ".[tpu]" -q

Cloning into 'maxtext'...
remote: Enumerating objects: 97903, done.
remote: Counting objects: 100% (535/535), done.
remote: Compressing objects: 100% (255/255), done.
remote: Total 97903 (delta 343), reused 428 (delta 280), pack-reused 97368 (from 2)
Receiving objects: 100% (97903/97903), 449.25 MiB | 46.47 MiB/s, done.
Resolving deltas: 100% (71844/71844), done.
/content/maxtext
Note: switching to 'maxtext-v0.2.2'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD is now at 348355c51 Update MaxText version to v0.2.2 and a

In [ ]:
# CELL 2 — Remove conflicting torch + verify
!pip uninstall torch torchvision torchaudio -y

%cd /content/maxtext/src
!python -c "import jax; print('Devices:', jax.devices())"
# Should show TpuDevice

# CELL 3 — Dirs
!mkdir -p /content/maxtext/runs
!mkdir -p /content/maxtext/logs

# CELL 4 — Train 0.71B custom scaled model
!python -u maxtext/trainers/pre_train/train.py \
  maxtext/configs/models/qwen3-0.6b.yml \
  steps=50 \
  dataset_type=synthetic \
  hardware=tpu \
  attention=dot_product \
  skip_jax_distributed_system=true \
  base_emb_dim=1152 \
  base_mlp_dim=3456 \
  base_output_directory=/content/maxtext/runs \
  run_name=qwen3_0p71b_tpu_50steps \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  micro_batch_size_to_train_on=1 \
  max_target_length=32 \
  max_prefill_predict_length=32 \
  enable_checkpointing=false \
  scan_layers=false \
  log_period=1 \
  weight_dtype=bfloat16 \
  2>&1 | tee /content/maxtext/logs/qwen3_0p71b_tpu.log

Found existing installation: torch 2.9.0+cpu
Uninstalling torch-2.9.0+cpu:
  Successfully uninstalled torch-2.9.0+cpu
Found existing installation: torchvision 0.24.0+cpu
Uninstalling torchvision-0.24.0+cpu:
  Successfully uninstalled torchvision-0.24.0+cpu
Found existing installation: torchaudio 2.9.0+cpu
Uninstalling torchaudio-2.9.0+cpu:
  Successfully uninstalled torchaudio-2.9.0+cpu
/content/maxtext/src
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TP